In [28]:
import os
os.chdir('/home/pedro/Documents/GIT_WORKSPACE/split_offdb/')
import geopandas as gpd
from shapely.geometry import MultiLineString, LineString, Polygon, MultiPolygon, LinearRing
from shapely.ops import split, linemerge, polygonize
import pandas as pd
import logging
import json
import time
import shapely as shp
from multiprocessing import Pool
from functools import partial
import numpy as np
from rtree import index
import psutil
from shapely.strtree import STRtree
from sqlalchemy import text, create_engine
from dotenv import load_dotenv
from sqlalchemy import Table, MetaData, Index
from split import Splitter
from sqlalchemy.dialects.postgresql import ARRAY, TEXT, INTEGER, NUMERIC, BIGINT

# Carregar variáveis do .env para conexão com o banco
load_dotenv()
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")


engine = create_engine(
    f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)
engine

start_time = time.time()

with open("config.json", "r") as f:
    config = json.load(f)  


s = Splitter()
s.colunas_boleanas(engine)
s.create_table(engine)

In [29]:
s._intersection_sql(n_grid = 71, engine = engine)
s.prepare_split_line()
s.perform_split()
s.process_overlapping()
s.format_gdf_broken_glass()

'0.01'

In [30]:

# Essa funcao precisa ser melhor pensada, pois aqui é o momento de facilitar as queries. Então, em cada rodada é bom poder 
# manipular livremente a saída.

# Formata o gdf broken glass, operações:
# 1. Dropa coluna ID
# 2. Drop onde é id_layer = ['GRID']
# 2. Formata os campos id_layer e id_feature para adequação ao db
# 3. Cria colunas booleanas e testa o campo id_layer para presenca da camada, retornando true ou false.
# 4. Calcula área das feicoes, apenas se drop_only_grid = True (default). Se for falso, mantem as feicoes id_layer=['GRID']
# 5. (TESTE) Incluir coluna com cd_uf e cd_mun  
# 6. Contagem de CAR


In [31]:
tabela_indice=pd.read_csv('/home/pedro/Documents/GIT_WORKSPACE/prod-Malha-Fundiaria/dados/tabela_indice.csv', sep = ';')
tabela_indice.loc[tabela_indice.categorias_fundiarias_v2025=='UCUS']

,hexadecimal,categorias_fundiarias_v2025
31355,702806,UCUS
31359,702800,UCUS
31611,702206,UCUS
31615,702200,UCUS
32379,700806,UCUS
32383,700800,UCUS
32635,700206,UCUS
32639,700200,UCUS
64123,102806,UCUS
64127,102800,UCUS


In [32]:
s.gdf_broken_glass.merge(tabela_indice, on=['hexadecimal','hexadecimal'], how = 'left')


,geometry,id_layer,id_feature,hexadecimal,cd_mun,cd_uf,n_car,is_uc,is_mun,is_car,is_ass,area_ha,id_layer_unico,categorias_fundiarias_v2025
0,"POLYGON ((-62 3.25389, -62.00052 3.25296, -62....","{GRID,UC,MUN}","{71,3375,1400027}",100200,1400027,14,0,True,True,False,False,28307.014326,"{GRID,MUN,UC}",UCUS
1,"POLYGON ((-62.00052 3.25296, -62 3.25389, -62 ...","{GRID,UC,MUN}","{71,3375,1400050}",100200,1400050,14,0,True,True,False,False,9028.303776,"{GRID,MUN,UC}",UCUS
2,"POLYGON ((-62.09085 3.24842, -62.09094 3.24831...","{GRID,MUN}","{71,1400027}",100000,1400027,14,0,False,True,False,False,0.045223,"{GRID,MUN}",ASRFG
3,"POLYGON ((-62.09094 3.24831, -62.09085 3.24842...","{GRID,MUN}","{71,1400050}",100000,1400050,14,0,False,True,False,False,104827.157309,"{GRID,MUN}",ASRFG
4,"POLYGON ((-62.0921 3.24694, -62.09138 3.24779,...","{GRID,UC,MUN}","{71,3375,1400050}",100200,1400050,14,0,True,True,False,False,0.042666,"{GRID,MUN,UC}",UCUS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,"POLYGON ((-62.00123 3.47749, -62.00125 3.47783...","{GRID,CAR,CAR,CAR,MUN,UC}","{71,3808669,5605330,5609933,1400027,3528}",100200,1400027,14,3,True,True,True,False,0.268544,"{CAR,GRID,MUN,UC}",UCUS
107,"POLYGON ((-62.00061 3.47618, -62.00118 3.47616...","{GRID,CAR,CAR,MUN,UC}","{71,3808669,5605330,1400027,3528}",100200,1400027,14,2,True,True,True,False,0.470403,"{CAR,GRID,MUN,UC}",UCUS
108,"POLYGON ((-62.0013 3.44681, -62.00114 3.42865,...","{GRID,CAR,MUN,UC}","{71,5605493,1400027,3528}",100200,1400027,14,1,True,True,True,False,998.033755,"{CAR,GRID,MUN,UC}",UCUS
109,"POLYGON ((-62.00204 3.5, -62.00947 3.49473, -6...","{GRID,CAR,MUN,UC}","{71,5605148,1400027,3528}",100200,1400027,14,1,True,True,True,False,30.881808,"{CAR,GRID,MUN,UC}",UCUS
